<a href="https://colab.research.google.com/github/adeniranlj-ui/projects1/blob/main/Text_Mining_ML_Practice.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Text Mining in Machine Learning

In [ ]:
# --- Setup & Imports ---
# Standard library
import zipfile  # read/write zip archives for toy corpora and embeddings

# Third-party libraries
import nltk  # tokenization and other NLP utilities
import numpy as np  # numerical computing
from nltk.stem.snowball import EnglishStemmer  # stemming
from nltk.tokenize import word_tokenize  # tokenization
from sklearn.decomposition import TruncatedSVD  # for LSA (SVD over TF-IDF)
from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer, ENGLISH_STOP_WORDS  # vectorization
from sklearn.linear_model import LogisticRegression  # classifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, roc_curve, auc  # evaluation
from sklearn.model_selection import train_test_split  # train/test split
from sklearn.pipeline import make_pipeline  # pipelines
from sklearn.preprocessing import Normalizer  # normalize vectors

# Download NLTK punkt model (needed for word_tokenize)
nltk.download('punkt')  # ensure the tokenizer data is present
nltk.download('punkt_tab') # Download punkt_tab as well, needed for some tokenization contexts

In [ ]:

# --- Lightweight helper functions to replace `mlba` utilities ---

def print_term_document_matrix(vectorizer, X, *, max_rows=10):
    """Pretty-print a few rows of a term-document (sparse) matrix."""
    # Get feature names (i.e., learned vocabulary terms)
    terms = vectorizer.get_feature_names_out()
    # Convert to dense array for quick display (limit rows)
    arr = X.toarray()
    rows_to_show = min(max_rows, arr.shape[0])
    print("Terms (first 20):", terms[:20])
    print("\nMatrix sample (first", rows_to_show, "rows):\n", arr[:rows_to_show])

def classification_summary(y_true, y_pred):
    """Print confusion matrix, accuracy, and classification report."""
    print("Confusion Matrix:\n", confusion_matrix(y_true, y_pred))
    print("\nAccuracy:", accuracy_score(y_true, y_pred))
    print("\nClassification Report:\n", classification_report(y_true, y_pred))


In [ ]:

# --- Toy Texts & Bag-of-Words with a custom token pattern ---

# Small demo corpus
text = [
    'this is the first     sentence!!',     # note punctuation and spaces
    'this is a second Sentence :)',         # note uppercase 'S' and a smiley
    'the third sentence, is here ',         # note trailing space and comma
    'forth of all sentences'                # intentionally misspelled 'fourth'
]

# Create a CountVectorizer that keeps alphabetic chars plus ! and :) as part of tokens
# \) means "match the literal parenthesis ) character."
count_vect = CountVectorizer(token_pattern=r'[a-zA-Z!:\)]+')  # allow letters, '!', and ')'
# It learns how to represent text numerically through a Bag-of-Words (BoW) representation using scikit-learn’s CountVectorizer
counts = count_vect.fit_transform(text)  # learn vocab and build term-document matrix
print_term_document_matrix(count_vect, counts)  # show a compact view


In [ ]:
# --- Custom Tokenizer: tokenize -> stem -> remove stop words ---

class StemmingTokenizer:
    # Initialize with a stemmer and English stop-word set
    def __init__(self):
        self.stemmer = EnglishStemmer()
        self.stopWords = set(ENGLISH_STOP_WORDS)
    # When called on a document, tokenize, filter, and stem
    def __call__(self, doc):
        # tokenize the string into tokens
        tokens = word_tokenize(doc)
        # keep alphabetic tokens not in stop words, then stem each token
        # self.stemmer.stem(t): Stemming consolidates similar word forms (e.g., run, running, runs → run), improving statistical efficiency.
        # isalpha() ensures we only use meaningful words (no punctuation or numbers). isalpha() is true if all characters in the string are alphabetic and there is at least one character
        return [self.stemmer.stem(t) for t in tokens if t.isalpha() and t.lower() not in self.stopWords]

# Learn features using the custom tokenizer
count_vect_custom = CountVectorizer(tokenizer=StemmingTokenizer(), token_pattern=None)
counts_custom = count_vect_custom.fit_transform(text)
print_term_document_matrix(count_vect_custom, counts_custom)  # display a small sample

In [ ]:

# --- CountVectorizer + TfidfTransformer (separate steps) ---

count_vect = CountVectorizer()  # default tokenization/processing
tfidfTransformer = TfidfTransformer(smooth_idf=False, norm=None)  # classic tf-idf, no norm for display
counts = count_vect.fit_transform(text)  # Bag-of-words model turns text documents into a term–document matrix
tfidf = tfidfTransformer.fit_transform(counts)  # transform to TF-IDF
print_term_document_matrix(count_vect, tfidf)  # show TF-IDF matrix (unnormalized) sample for all 12 terms in all 4 documents


In [ ]:
import zipfile
# --- Build a tiny zipped corpus to simulate 'AutoAndElectronics.zip' ---

tiny_corpus_zip = "/content/AutoAndElectronics.zip"  # path to save the toy zip

# Two mini documents per class for demonstration
autos_docs = [
    "I love my new sedan. The engine performance is smooth and quiet.",
    "Changing oil and rotating tires regularly keeps a car reliable."
]
electronics_docs = [
    "The new smartphone has a bright OLED display and long battery life.",
    "Upgrading a graphics card improves frame rates for gaming."
]

# Create the zip with two folders mimicking 'rec.autos' and 'sci.electronics'
with zipfile.ZipFile(tiny_corpus_zip, 'w') as zf:
    for i, doc in enumerate(autos_docs, start=1):
        zf.writestr(f'rec.autos/doc_{i}.txt', doc)
    for i, doc in enumerate(electronics_docs, start=1):
        zf.writestr(f'sci.electronics/doc_{i}.txt', doc)

# Read corpus and labels from the zip (1 for autos, 0 for electronics, matching the original intent)
corpus = []  # list of raw texts
label = []   # 1 = rec.autos, 0 = sci.electronics
with zipfile.ZipFile(tiny_corpus_zip) as rawData:
    for info in rawData.infolist():
        if info.is_dir():
            continue
        # label by folder name in the file path
        label.append(1 if 'rec.autos' in info.filename else 0)
        corpus.append(rawData.read(info).decode('utf-8'))

len(corpus), sum(label)  # quick sanity check: (#docs, #autos)
# There are 4 documents total, and 2 of them belong to the “rec.autos” category.

In [ ]:

# --- Vectorize the corpus with the custom tokenizer, then TF-IDF ---

preprocessor = CountVectorizer(tokenizer=StemmingTokenizer(), token_pattern=None)  # custom tokenizer
preprocessedText = preprocessor.fit_transform(corpus)  # bag-of-words over the corpus

tfidfTransformer = TfidfTransformer()  # standard tf-idf
tfidf = tfidfTransformer.fit_transform(preprocessedText)  # TF-IDF features
tfidf.shape  # (#docs, #terms)


In [ ]:
# --- Latent Semantic Analysis (LSA) on TF-IDF + Logistic Regression ---

# Use 2 concepts for this tiny dataset (keep it small to avoid overfitting in the demo)
# copy=False tells scikit-learn to perform normalization in place rather than making a copy of the data.
lsa = make_pipeline(TruncatedSVD(2, random_state=123), Normalizer(copy=False))  # LSA pipeline
lsa_tfidf = lsa.fit_transform(tfidf)  # project TF-IDF into concept space

# Train/test split (60% train / 40% test)
Xtrain, Xtest, ytrain, ytest = train_test_split(lsa_tfidf, label, test_size=0.4, random_state=42)

# Logistic regression classifier
logit_reg = LogisticRegression(solver='lbfgs')  # simple linear classifier
logit_reg.fit(Xtrain, ytrain)  # fit the model on training data

# Predictions on the test set
y_pred = logit_reg.predict(Xtest)  # discrete predictions
y_scores = logit_reg.predict_proba(Xtest)[:, 1]  # positive-class probabilities

# Evaluation summary (replace mlba.classificationSummary)
classification_summary(y_true=ytest, y_pred=y_pred)

# ROC curve and AUC for reference
fpr, tpr, _ = roc_curve(ytest, y_scores)
print("AUC:", auc(fpr, tpr))
#              Predicted: 0	      Predicted: 1
# Actual: 0	True Negative (TN)	False Positive (FP)
# Actual: 1	False Negative (FN)	True Positive (TP)
# AUC measure the probability that the model ranks a randomly chosen positive instance higher than a randomly chosen negative instance.
# AUC = 0.5 means model perform the same as random guessing
# A model needs to reach at least 0.7 in AUC to be useful

In [ ]:
# ===================== BERT fine-tuning (binary classification) =====================

!pip -q install transformers accelerate

import numpy as np
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, accuracy_score, classification_report, roc_curve, auc
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    set_seed
)
from datasets import Dataset

# 0) Reproducibility and device -------------------------------------------------------
set_seed(123)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 1) Train/test split on raw text -----------------------------------------------------
y = np.array(label)
Xtrain_text, Xtest_text, ytrain, ytest = train_test_split(
    corpus, y, test_size=0.4, random_state=42, stratify=y if len(set(y)) > 1 else None
)

# 2) Tokenizer and HF Datasets --------------------------------------------------------
model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_batch(batch):
    return tokenizer(
        batch["text"],
        # Padding means adding extra blank tokens to shorter text sequences so that all sequences in a batch have the same length.
        # Padding is important because most deep learning models require inputs within a batch to have the same length so they can be processed efficiently in parallel.
        padding=False,          # we'll pad dynamically with the collator
        truncation=True,
        max_length=256
    )

train_ds = Dataset.from_dict({"text": Xtrain_text, "label": ytrain.tolist()})
test_ds  = Dataset.from_dict({"text": Xtest_text,  "label": ytest.tolist()})
# remove_columns=["text"] means that after applying tokenize_batch to the dataset, the original "text" column is deleted from test_ds, keeping only the new tokenized features (e.g., input_ids, attention_mask) in the dataset.
train_ds = train_ds.map(tokenize_batch, batched=True, remove_columns=["text"])
test_ds  = test_ds.map(tokenize_batch,  batched=True, remove_columns=["text"])

# Collator creates a batch-processing function that dynamically pads tokenized sequences to the same length within each batch during training or evaluation.
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# 3) Model ---------------------------------------------------------------------------
# Defines a dictionary mapping numeric class indices to human-readable label names
id2label = {0: "sci.electronics", 1: "rec.autos"}
# Defines a reverse dictionary mapping label names to their corresponding numeric class indices
label2id = {"sci.electronics": 0, "rec.autos": 1}

# Loads a pretrained transformer model for sequence classification based on the specified model name
model = AutoModelForSequenceClassification.from_pretrained(
    # Specifies which pretrained model architecture and weights to load (e.g., "bert-base-uncased")
    model_name,
    # Sets the number of output classes for the classification head
    num_labels=2,
    # Provides a mapping from numeric class IDs to human-readable label names
    id2label=id2label,
    # Provides a mapping from label names back to numeric class IDs
    label2id=label2id
# Moves the model to the selected computation device (GPU if available, otherwise CPU)
).to(device)

# 4) Training arguments ---------------------------------------------------------------
training_args = TrainingArguments(
    output_dir="./bert_cls",
    learning_rate=2e-5,
    # Defines the batch size per device (GPU/CPU) during training
    per_device_train_batch_size=8,
    # Defines the batch size per device during evaluation
    per_device_eval_batch_size=8,
    # Sets the number of full passes over the training dataset
    num_train_epochs=3,                 # small demo; increase for real data
    # Applies L2 weight decay regularization to reduce overfitting, weight decay = 0 means no regularization, weight decay = 0.1 is already very big
    weight_decay=0.01,
    eval_strategy="epoch", # Changed from evaluation_strategy
    logging_strategy="steps",
    # Logs metrics every specified number of training steps
    logging_steps=10,
    # Saves a model checkpoint at the end of each epoch
    save_strategy="epoch",
    # Automatically reloads the best-performing model (based on chosen metric) after training
    load_best_model_at_end=True,
    # Specifies which evaluation metric determines the best model
    metric_for_best_model="eval_loss",
    report_to=[]                        # disable wandb if present
)

# Optional: simple metrics for Trainer, it tells the Trainer how to calculate and report accuracy after each evaluation phase.
def compute_metrics(eval_pred):
    preds, labels = eval_pred
    if isinstance(preds, tuple):  # newer HF versions pass (logits,)
        preds = preds[0]
    # Converts raw logits into predicted class indices by selecting the index with the highest score
    y_pred = preds.argmax(axis=-1)
    # Computes classification accuracy by comparing predicted labels with true labels
    acc = (y_pred == labels).mean()
    return {"accuracy": float(acc)}

# 5) Trainer -------------------------------------------------------------------------
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

trainer.train()

# 6) Evaluation (confusion matrix, report, AUC) --------------------------------------
# Disables gradient computation to reduce memory usage and speed up inference during evaluation
with torch.no_grad():
    # Uses the Trainer to generate predictions (logits) for the test dataset
    logits = trainer.predict(test_ds).predictions
# Converts raw model logits into predicted class indices by selecting the highest-scoring class for each sample
y_pred = logits.argmax(axis=-1)

print("Confusion matrix:\n", confusion_matrix(ytest, y_pred))
print("\nAccuracy:", accuracy_score(ytest, y_pred))
print("\nClassification report:\n", classification_report(ytest, y_pred, target_names=["sci.electronics", "rec.autos"]))

# ROC/AUC (only meaningful if both classes appear in ytest)
try:
    import warnings
    from sklearn.exceptions import UndefinedMetricWarning
    warnings.filterwarnings("ignore", category=UndefinedMetricWarning)
    y_scores = torch.tensor(logits).softmax(dim=-1).numpy()[:, 1]
    if len(set(ytest)) == 2:
        fpr, tpr, _ = roc_curve(ytest, y_scores)
        print("AUC:", auc(fpr, tpr))
    else:
        print("AUC: unavailable (test set has a single class)")
except Exception as e:
    print("AUC not computed:", e)
# =================================================================================================


In [ ]:

# --- Tiny GloVe-like embeddings + Sentence Embedding Vectorizer demo ---
# GloVe, which stands for Global Vectors for Word Representation, is an algorithm for creating word embeddings (numerical representations of words)
# https://nlp.stanford.edu/projects/glove/
# https://mmuratarat.github.io/2020-03-20/glove
# Create a very small GloVe-like file (word + 5-dim vector) packed in a zip
glove_zip_path = '/content/glove.6B.zip'
glove_txt_name = 'glove.6B.5d.txt'
# Creates a small multiline string simulating GloVe word embeddings (word + 5-dimensional vector)
toy_glove = """
the 0.1 0.2 0.3 0.4 0.5
new 0.05 -0.1 0.2 -0.05 0.3
car 0.2 0.0 0.1 0.3 -0.2
phone -0.1 0.2 -0.2 0.1 0.0
engine 0.3 0.1 0.0 0.2 0.1
battery -0.2 0.3 0.1 0.0 0.2
gaming 0.1 -0.2 0.3 0.2 -0.1
display -0.05 0.1 0.05 0.0 0.15
oil 0.15 0.05 0.0 0.1 -0.05
tires 0.12 0.04 -0.02 0.08 0.01
""".strip()  # keep it tiny for class demo

# Creates a zip file and writes the toy GloVe text content into it
with zipfile.ZipFile(glove_zip_path, 'w') as zf:
    zf.writestr(glove_txt_name, toy_glove)

# Load embeddings from the zip into a dict
embeddings = {}
with (zipfile.Path(glove_zip_path) / glove_txt_name).open('rt') as f:
    for line in f:
        # Splits the line into tokens: first token is word, remaining are vector values
        parts = line.strip().split()
        # # Converts vector values to float32 numpy array and stores in dictionary
        embeddings[parts[0]] = np.asarray(parts[1:], dtype='float32')

# Define a simple sentence embedding vectorizer (average of word embeddings)
class SentenceEmbeddingVectorizer:
    # Initializes the vectorizer with a preloaded embedding dictionary and embedding dimension
    def __init__(self, embeddings, *, dim=5):
        self.embeddings = embeddings
        self.dim = dim
    # Fits and transforms input sentences into embedding vectors
    def fit_transform(self, X, y=None):
        self.fit(X)
        return self.transform(X)
    def fit(self, X, y=None):
        # No fitting needed for averaging pre-trained embeddings
        return self
    # Converts a list of sentences into fixed-length numeric vectors
    def transform(self, X):
        # Stores resulting sentence vectors
        vectors = []
        for sentence in X:
            # Splits sentence into individual words
            words = sentence.split()
            # Retrieves embedding for each word; if not found, substitutes zero vector
            vecs = [self.embeddings.get(w.lower(), np.zeros(self.dim)) for w in words]
            if len(vecs) == 0:
                vectors.append(np.zeros(self.dim))
            else:
                vectors.append(np.mean(vecs, axis=0))
        # Converts list of sentence vectors into numpy array
        return np.asarray(vectors)

# Instantiates the vectorizer and transforms our tiny corpus into sentence embeddings
sent_vec = SentenceEmbeddingVectorizer(embeddings, dim=5).fit_transform(corpus)
sent_vec[:2]  # show first two vectors, which represents the first two sentences/documents,
